In [1]:
from dataclasses import dataclass
from pathlib import Path
import gc
import importlib.util
import random
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
try:
    from huggingface_hub.utils import disable_progress_bars

    disable_progress_bars()
except Exception:
    pass

import forgi
import forgi.visual.mplotlib as fvmplot
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from datasets import load_dataset
from IPython.display import display
from sklearn.metrics import f1_score as sklearn_f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


def configure_cuda_tensor_cores():
    if not torch.cuda.is_available():
        return
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = False
def print_tensor_core_status():
    if not torch.cuda.is_available():
        print("CUDA unavailable; tensor cores will not be used.")
        return
    capability = torch.cuda.get_device_capability()
    print(f"CUDA device: {torch.cuda.get_device_name(0)} | capability={capability}")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")
    print(f"TF32 matmul enabled: {torch.backends.cuda.matmul.allow_tf32}")
    print(f"TF32 cuDNN enabled: {torch.backends.cudnn.allow_tf32}")
    print(f"float32 matmul precision: {torch.get_float32_matmul_precision()}")


def print_attention_backend_status():
    print(f"flash-attn package installed: {importlib.util.find_spec('flash_attn') is not None}")
    print(f"PyTorch SDPA available: {hasattr(F, 'scaled_dot_product_attention')}")
    if torch.cuda.is_available():
        print(f"PyTorch flash SDPA enabled: {torch.backends.cuda.flash_sdp_enabled()}")
        print(f"PyTorch memory-efficient SDPA enabled: {torch.backends.cuda.mem_efficient_sdp_enabled()}")
        print(f"PyTorch math SDPA enabled: {torch.backends.cuda.math_sdp_enabled()}")


def release_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


configure_cuda_tensor_cores()

np.set_printoptions(threshold=np.inf, linewidth=1000)


In [2]:
HF_DATASET_NAME = "multimolecule/bprna-spot-0"
raw_dataset = load_dataset(HF_DATASET_NAME)
print(f"Loaded {HF_DATASET_NAME}")
for split_name, split_dataset in raw_dataset.items():
    print(f"{split_name}: {len(split_dataset):,} rows")


Loaded multimolecule/bprna-spot-0
train: 10,814 rows
validation: 1,300 rows
test: 1,305 rows


In [3]:
OPEN_TO_CLOSE = {"(": ")", "[": "]", "{": "}", "<": ">"}
OPEN_TO_CLOSE.update({chr(ord("A") + i): chr(ord("a") + i) for i in range(26)})
CLOSE_TO_OPEN = {close: open_ for open_, close in OPEN_TO_CLOSE.items()}


@dataclass(frozen=True)
class DBNRecord:
    name: str
    path: Path
    sequence: str
    structure: str
    contacts: tuple[tuple[int, int, str], ...]


def parse_contacts(structure: str) -> tuple[tuple[int, int, str], ...]:
    stacks = {open_: [] for open_ in OPEN_TO_CLOSE}
    contacts = []

    for position, char in enumerate(structure):
        if char in OPEN_TO_CLOSE:
            stacks[char].append(position)
        elif char in CLOSE_TO_OPEN:
            open_char = CLOSE_TO_OPEN[char]
            if not stacks[open_char]:
                raise ValueError(f"Unmatched closing symbol {char!r} at position {position + 1}")
            start = stacks[open_char].pop()
            contacts.append((start, position, open_char + char))
        elif char == ".":
            continue
        else:
            raise ValueError(f"Unsupported dot-bracket symbol {char!r}")

    unmatched = {char: positions for char, positions in stacks.items() if positions}
    if unmatched:
        preview = {char: [p + 1 for p in positions[:5]] for char, positions in unmatched.items()}
        raise ValueError(f"Unmatched opening symbols: {preview}")

    return tuple(sorted(contacts))


def read_dbn(path: Path) -> DBNRecord:
    lines = [line.strip() for line in path.read_text().splitlines() if line.strip()]
    data_lines = [line for line in lines if not line.startswith("#")]
    if len(data_lines) < 2:
        raise ValueError(f"Expected sequence and structure lines in {path}")

    sequence, structure = data_lines[:2]
    if len(sequence) != len(structure):
        raise ValueError(
            f"Sequence/structure length mismatch in {path.name}: {len(sequence)} != {len(structure)}"
        )

    name = path.stem
    for line in lines:
        if line.startswith("#Name:"):
            name = line.split(":", 1)[1].strip()
            break

    return DBNRecord(name=name, path=path, sequence=sequence, structure=structure, contacts=parse_contacts(structure))


def record_from_hf_row(row, split_name: str) -> DBNRecord:
    sequence = row["sequence"].strip()
    structure = row["secondary_structure"].strip()
    if len(sequence) != len(structure):
        raise ValueError(
            f"Sequence/structure length mismatch in {row['id']}: {len(sequence)} != {len(structure)}"
        )
    return DBNRecord(
        name=row["id"],
        path=Path(f"{split_name}/{row['id']}.dbn"),
        sequence=sequence,
        structure=structure,
        contacts=parse_contacts(structure),
    )


def build_contact_map(record: DBNRecord, device=None, dtype=torch.float32) -> torch.Tensor:
    matrix = torch.zeros((len(record.sequence), len(record.sequence)), dtype=dtype, device=device)
    for i, j, _ in record.contacts:
        matrix[i, j] = 1
        matrix[j, i] = 1
    return matrix


In [4]:
hf_records_by_split = {
    split_name: [record_from_hf_row(row, split_name) for row in split_dataset]
    for split_name, split_dataset in raw_dataset.items()
}
train_records = hf_records_by_split["train"]
validation_records = hf_records_by_split["validation"]
test_records = hf_records_by_split["test"]
all_records = train_records + validation_records + test_records

for split_name, split_records in hf_records_by_split.items():
    print(f"Parsed {split_name}: {len(split_records):,} records")
print(f"Parsed total: {len(all_records):,} records")


Parsed train: 10,814 records
Parsed validation: 1,300 records
Parsed test: 1,305 records
Parsed total: 13,419 records


In [5]:
def load_record_as_forgi(record: DBNRecord):
    import tempfile

    with tempfile.NamedTemporaryFile("w", suffix=".fa", delete=False) as handle:
        handle.write(f">{record.name}\n{record.sequence}\n{record.structure}\n")
        temp_path = Path(handle.name)

    try:
        return forgi.load_rna(str(temp_path), allow_many=False)
    finally:
        temp_path.unlink(missing_ok=True)


def draw_secondary_structure(record: DBNRecord, ax=None, title=None, contact_color="#c92a2a"):
    cg = load_record_as_forgi(record)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7), dpi=140)
    basepair_kwargs = {"linewidth": 0.0, "alpha": 0.0}

    # Suppress optional residue-number arrows; compact layouts can produce invalid arrow geometry.
    original_number_annotator = fvmplot._find_annot_pos_on_circle
    fvmplot._find_annot_pos_on_circle = lambda *args, **kwargs: None
    try:
        _, coords = fvmplot.plot_rna(
            cg,
            ax=ax,
            text_kwargs={"fontsize": 5},
            backbone_kwargs={"linewidth": 1.2, "color": "#2b2f33"},
            basepair_kwargs=basepair_kwargs,
            lighten=0.35,
            annotations=None,
        )
    finally:
        fvmplot._find_annot_pos_on_circle = original_number_annotator

    for contact in record.contacts:
        i, j = contact[:2]
        ax.plot(
            [coords[i, 0], coords[j, 0]],
            [coords[i, 1], coords[j, 1]],
            color=contact_color,
            linewidth=1.1,
            alpha=0.72,
            zorder=0,
        )

    ax.set_title(
        title or f"{record.name}\nlength={len(record.sequence)}, contacts={len(record.contacts)}",
        fontsize=11,
    )
    return ax

BRACKET_LEVELS = ["()", "[]", "{}", "<>"] + [chr(ord("A") + i) + chr(ord("a") + i) for i in range(26)]
CANONICAL_BASE_PAIRS = {("A", "U"), ("U", "A"), ("G", "C"), ("C", "G"), ("G", "U"), ("U", "G")}


def build_canonical_pair_mask(sequence: str, device=None):
    sequence = sequence.upper()
    base_to_id = {"A": 0, "C": 1, "G": 2, "U": 3, "T": 3}
    base_ids = torch.tensor([base_to_id.get(base, -1) for base in sequence], device=device)
    left = base_ids[:, None]
    right = base_ids[None, :]
    mask = (
        ((left == 0) & (right == 3))
        | ((left == 3) & (right == 0))
        | ((left == 2) & (right == 1))
        | ((left == 1) & (right == 2))
        | ((left == 2) & (right == 3))
        | ((left == 3) & (right == 2))
    )
    mask.fill_diagonal_(False)
    return mask


def filter_contacts_by_pair_mask(contacts, valid_pair_mask=None):
    if valid_pair_mask is None:
        return tuple(contacts)
    mask = valid_pair_mask.detach().bool().cpu().numpy() if torch.is_tensor(valid_pair_mask) else valid_pair_mask
    return tuple(contact for contact in contacts if mask[int(contact[0]), int(contact[1])])


def constrained_contact_map_from_logits(logits, L=None, th=0.5, valid_pair_mask=None):
    logits = logits.detach().float()
    L = logits.shape[0] if L is None else L
    prob = torch.sigmoid(logits[:L, :L]).clone()
    prob.fill_diagonal_(0.0)
    if valid_pair_mask is not None:
        valid_pair_mask = valid_pair_mask.detach().to(device=prob.device, dtype=torch.bool)[:L, :L]
        prob = prob.masked_fill(~valid_pair_mask, 0.0)

    row_index, column_index = torch.triu_indices(L, L, offset=1, device=prob.device)
    scores = prob[row_index, column_index]
    candidate_positions = torch.nonzero(scores > th, as_tuple=False).flatten()
    pred = torch.zeros((L, L), dtype=torch.bool, device=prob.device)
    if candidate_positions.numel() == 0:
        return pred

    candidate_scores = scores[candidate_positions].detach().cpu().numpy()
    candidate_i = row_index[candidate_positions].detach().cpu().numpy()
    candidate_j = column_index[candidate_positions].detach().cpu().numpy()
    used = np.zeros(L, dtype=bool)
    for candidate_index in np.argsort(-candidate_scores):
        i = int(candidate_i[candidate_index])
        j = int(candidate_j[candidate_index])
        if used[i] or used[j]:
            continue
        pred[i, j] = True
        pred[j, i] = True
        used[i] = True
        used[j] = True
    return pred


def predicted_contacts_from_logits(logits, th=0.5, valid_pair_mask=None):
    logits = logits.detach().float()
    L = logits.shape[0]
    pred = constrained_contact_map_from_logits(logits, L=L, th=th, valid_pair_mask=valid_pair_mask)
    prob = torch.sigmoid(logits[:L, :L]).detach().cpu().numpy()
    pred_cpu = pred.detach().cpu().numpy()
    return [(i, j, float(prob[i, j])) for i, j in zip(*np.triu(pred_cpu, k=1).nonzero())]


def f1_rna_llm_folding(ref, logits, L, th=0.5, valid_pair_mask=None):
    ref = ref[:L, :L]
    pred = constrained_contact_map_from_logits(logits, L=L, th=th, valid_pair_mask=valid_pair_mask)
    i, j = torch.triu_indices(L, L, offset=1, device=ref.device)
    pair_mask = torch.ones_like(i, dtype=torch.bool)
    if valid_pair_mask is not None:
        valid_pair_mask = valid_pair_mask.detach().to(device=ref.device, dtype=torch.bool)[:L, :L]
        pair_mask = valid_pair_mask[i, j]
    y_true = ref[i, j][pair_mask].detach().cpu().numpy().ravel()
    y_pred = pred.to(device=ref.device)[i, j][pair_mask].detach().cpu().numpy().ravel()
    if y_true.size == 0:
        return 0.0
    return sklearn_f1_score(y_true, y_pred, zero_division=0)


def f1_threshold_sweep_from_logits(ref, logits, L, thresholds, valid_pair_mask=None):
    ref = ref[:L, :L]
    prob = torch.sigmoid(logits[:L, :L]).detach()
    i, j = torch.triu_indices(L, L, offset=1, device=prob.device)
    pair_mask = torch.ones_like(i, dtype=torch.bool)
    if valid_pair_mask is not None:
        valid_pair_mask = valid_pair_mask.detach().to(device=prob.device, dtype=torch.bool)[:L, :L]
        pair_mask = valid_pair_mask[i, j]
    y_true = ref[i, j][pair_mask].detach().cpu().numpy().ravel()
    scores = prob[i, j][pair_mask].detach().cpu().numpy().ravel()
    candidate_i = i[pair_mask].detach().cpu().numpy()
    candidate_j = j[pair_mask].detach().cpu().numpy()
    candidate_order = np.argsort(-scores)
    if y_true.size == 0:
        return {threshold: 0.0 for threshold in thresholds}

    f1_by_threshold = {}
    for threshold in thresholds:
        y_pred = np.zeros_like(y_true, dtype=bool)
        used_positions = np.zeros(L, dtype=bool)
        for candidate_index in candidate_order:
            if scores[candidate_index] <= threshold:
                break
            left = int(candidate_i[candidate_index])
            right = int(candidate_j[candidate_index])
            if used_positions[left] or used_positions[right]:
                continue
            y_pred[candidate_index] = True
            used_positions[left] = True
            used_positions[right] = True
        f1_by_threshold[threshold] = sklearn_f1_score(y_true, y_pred, zero_division=0)
    return f1_by_threshold


def contacts_to_dotbracket(length: int, contacts):
    structure = ["."] * length
    selected_contacts = []
    used_positions = set()
    contacts_by_level = {level: [] for level in range(len(BRACKET_LEVELS))}

    def crosses(first, second):
        a, b = first
        c, d = second
        return (a < c < b < d) or (c < a < d < b)

    scored_contacts = []
    for contact in contacts:
        i, j = int(contact[0]), int(contact[1])
        score = float(contact[2]) if len(contact) > 2 and isinstance(contact[2], (int, float, np.floating)) else 1.0
        if i == j:
            continue
        if i > j:
            i, j = j, i
        if 0 <= i < length and 0 <= j < length:
            scored_contacts.append((score, i, j))

    for score, i, j in sorted(scored_contacts, reverse=True):
        if i in used_positions or j in used_positions:
            continue

        assigned_level = None
        for level, existing_contacts in contacts_by_level.items():
            if all(not crosses((i, j), existing) for existing in existing_contacts):
                assigned_level = level
                break

        if assigned_level is None:
            continue

        open_symbol, close_symbol = BRACKET_LEVELS[assigned_level]
        structure[i] = open_symbol
        structure[j] = close_symbol
        contacts_by_level[assigned_level].append((i, j))
        selected_contacts.append((i, j, open_symbol + close_symbol))
        used_positions.update((i, j))

    return "".join(structure), tuple(sorted(selected_contacts))

def show_true_and_predicted_structures(record: DBNRecord, logits, valid_pair_mask=None, th=0.5):
    true_contacts = filter_contacts_by_pair_mask(record.contacts, valid_pair_mask)
    true_record = DBNRecord(
        name=record.name,
        path=record.path,
        sequence=record.sequence,
        structure=record.structure,
        contacts=true_contacts,
    )
    thresholded_contacts = predicted_contacts_from_logits(logits, th=th, valid_pair_mask=valid_pair_mask)
    predicted_structure, predicted_contacts = contacts_to_dotbracket(len(record.sequence), thresholded_contacts)
    predicted_record = DBNRecord(
        name=f"{record.name}_predicted",
        path=record.path,
        sequence=record.sequence,
        structure=predicted_structure,
        contacts=predicted_contacts,
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 6), dpi=140)
    draw_secondary_structure(
        true_record,
        ax=axes[0],
        title=f"True: {record.name}\ncontacts={len(true_record.contacts)} / original={len(record.contacts)}",
        contact_color="#2f9e44",
    )
    draw_secondary_structure(
        predicted_record,
        ax=axes[1],
        title=f"Predicted p >= {th}\ncontacts={len(predicted_record.contacts)} / thresholded={len(thresholded_contacts)}",
        contact_color="#c92a2a",
    )
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def show_f1_history(f1_values, title="Training F1"):
    values = np.asarray(f1_values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        print("No F1 values to plot yet.")
        return

    lower = float(np.quantile(values, 0.1))
    upper = float(values.max())
    if upper <= lower:
        upper = min(1.0, lower + 0.01)
        lower = max(0.0, lower - 0.01)

    fig, ax = plt.subplots(figsize=(8, 3.5), dpi=140)
    ax.plot(np.arange(1, len(f1_values) + 1), f1_values, linewidth=1.3)
    ax.set_title(title)
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("F1")
    ax.set_ylim(lower, upper)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


In [6]:
class RNADotBracketDataset(Dataset):
    def __init__(self, records, max_length=500, min_contacts=1):
        self.records = [
            record
            for record in records
            if len(record.sequence) < max_length and len(record.contacts) >= min_contacts
        ]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        return {
            "record": record,
            "sequence": record.sequence,
            "target_adjacency": build_contact_map(record),
        }


def collate_single_record(batch):
    if len(batch) != 1:
        raise ValueError("This notebook uses batch_size=1 for variable-size NxN targets")
    return batch[0]



In [7]:
class GRN(nn.Module):
    def __init__(self, channels, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(channels))
        self.beta = nn.Parameter(torch.zeros(channels))
        self.eps = eps

    def forward(self, x):
        spatial_dims = tuple(range(1, x.ndim - 1))
        response = torch.norm(x, p=2, dim=spatial_dims, keepdim=True)
        normalized_response = response / (response.mean(dim=-1, keepdim=True) + self.eps)
        return self.gamma * (x * normalized_response) + self.beta + x


class ConvNeXtV2Block(nn.Module):
    def __init__(self, channels, expansion=4):
        super().__init__()
        expanded_channels = expansion * channels
        self.depthwise_conv = nn.Conv2d(channels, channels, kernel_size=7, padding=3, groups=channels)
        self.norm = nn.LayerNorm(channels)
        self.pointwise_expand = nn.Linear(channels, expanded_channels)
        self.grn = GRN(expanded_channels)
        self.pointwise_project = nn.Linear(expanded_channels, channels)

    def forward(self, x):
        residual = x
        x = self.depthwise_conv(x)
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = self.pointwise_expand(x)
        x = F.gelu(x)
        x = self.grn(x)
        x = self.pointwise_project(x)
        x = x.permute(0, 3, 1, 2)
        return residual + x


class ConvNeXtV2Block1D(nn.Module):
    def __init__(self, channels, expansion=4):
        super().__init__()
        expanded_channels = expansion * channels
        self.depthwise_conv = nn.Conv1d(channels, channels, kernel_size=7, padding=3, groups=channels)
        self.norm = nn.LayerNorm(channels)
        self.pointwise_expand = nn.Linear(channels, expanded_channels)
        self.grn = GRN(expanded_channels)
        self.pointwise_project = nn.Linear(expanded_channels, channels)

    def forward(self, x):
        residual = x
        x = self.depthwise_conv(x.transpose(1, 2).contiguous()).transpose(1, 2)
        x = self.norm(x)
        x = self.pointwise_expand(x)
        x = F.gelu(x)
        x = self.grn(x)
        x = self.pointwise_project(x)
        return residual + x


RNA_BASE_TO_VECTOR = {
    "A": (1.0, 0.0, 0.0, 0.0),
    "C": (0.0, 1.0, 0.0, 0.0),
    "G": (0.0, 0.0, 1.0, 0.0),
    "U": (0.0, 0.0, 0.0, 1.0),
    "T": (0.0, 0.0, 0.0, 1.0),
    "R": (0.5, 0.0, 0.5, 0.0),
    "Y": (0.0, 0.5, 0.0, 0.5),
    "S": (0.0, 0.5, 0.5, 0.0),
    "W": (0.5, 0.0, 0.0, 0.5),
    "K": (0.0, 0.0, 0.5, 0.5),
    "M": (0.5, 0.5, 0.0, 0.0),
    "B": (0.0, 1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0),
    "D": (1.0 / 3.0, 0.0, 1.0 / 3.0, 1.0 / 3.0),
    "H": (1.0 / 3.0, 1.0 / 3.0, 0.0, 1.0 / 3.0),
    "V": (1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0, 0.0),
    "N": (0.25, 0.25, 0.25, 0.25),
    "X": (0.25, 0.25, 0.25, 0.25),
}


def sequence_to_one_hot(sequence: str, device, dtype=torch.float32):
    base_vectors = [RNA_BASE_TO_VECTOR.get(base, RNA_BASE_TO_VECTOR["N"]) for base in sequence.upper()]
    return torch.tensor(base_vectors, device=device, dtype=dtype)


PAIR_METADATA_BASES = ("A", "C", "G", "U")
PAIR_METADATA_BASE_TO_INDEX = {base: index for index, base in enumerate(PAIR_METADATA_BASES)}
PAIR_METADATA_BASE_TO_INDEX["T"] = PAIR_METADATA_BASE_TO_INDEX["U"]
UNORDERED_BASE_PAIR_TYPES = tuple(
    (left_base, right_base)
    for left_index, left_base in enumerate(PAIR_METADATA_BASES)
    for right_base in PAIR_METADATA_BASES[left_index:]
)


def relative_position_sinusoidal_features(sequence_length, feature_dim, device, dtype=torch.float32):
    if feature_dim <= 0:
        return torch.empty((sequence_length, sequence_length, 0), device=device, dtype=dtype)
    positions = torch.arange(sequence_length, device=device, dtype=torch.float32)
    relative_positions = positions[:, None] - positions[None, :]
    half_dim = (feature_dim + 1) // 2
    inv_freq = 1.0 / (10000 ** (torch.arange(half_dim, device=device, dtype=torch.float32) / half_dim))
    angles = relative_positions[..., None] * inv_freq
    features = torch.cat([angles.sin(), angles.cos()], dim=-1)[..., :feature_dim]
    return features.to(dtype=dtype)


def unordered_base_pair_one_hot_features(sequence: str, device, dtype=torch.float32):
    base_indices = [PAIR_METADATA_BASE_TO_INDEX.get(base, -1) for base in sequence.upper()]
    base_indices = torch.tensor(base_indices, device=device, dtype=torch.long)
    left_indices = base_indices[:, None]
    right_indices = base_indices[None, :]
    valid_pairs = (left_indices >= 0) & (right_indices >= 0)
    lower_indices = torch.minimum(left_indices.clamp_min(0), right_indices.clamp_min(0))
    upper_indices = torch.maximum(left_indices.clamp_min(0), right_indices.clamp_min(0))
    pair_indices = (
        lower_indices * len(PAIR_METADATA_BASES)
        - lower_indices * (lower_indices - 1) // 2
        + (upper_indices - lower_indices)
    )
    features = F.one_hot(pair_indices, num_classes=len(UNORDERED_BASE_PAIR_TYPES)).to(dtype=dtype)
    return features * valid_pairs[..., None].to(dtype=dtype)


def rotate_half(x):
    left, right = x.chunk(2, dim=-1)
    return torch.cat([-right, left], dim=-1)


def add_noncanonical_logit_penalty(logits, valid_pair_mask, penalty=-3.0):
    if valid_pair_mask is None:
        return logits
    valid_pair_mask = valid_pair_mask.to(device=logits.device, dtype=torch.bool)
    if valid_pair_mask.shape != logits.shape:
        raise ValueError(f"Expected valid pair mask shape {tuple(logits.shape)}, got {tuple(valid_pair_mask.shape)}")
    return logits + (~valid_pair_mask).to(dtype=logits.dtype) * penalty


def add_short_range_logit_penalty(logits, max_distance=4, penalty=-3.0):
    if max_distance < 0 or penalty == 0:
        return logits
    positions = torch.arange(logits.shape[0], device=logits.device)
    short_range_mask = (positions[:, None] - positions[None, :]).abs() <= max_distance
    return logits + short_range_mask.to(dtype=logits.dtype) * penalty


def apply_rope(q, k):
    head_dim = q.shape[-1]
    if head_dim % 2 != 0:
        raise ValueError("RoPE requires an even attention head dimension")
    positions = torch.arange(q.shape[-2], device=q.device, dtype=torch.float32)
    inv_freq = 1.0 / (10000 ** (torch.arange(0, head_dim, 2, device=q.device, dtype=torch.float32) / head_dim))
    angles = torch.einsum("l,d->ld", positions, inv_freq)
    angles = torch.cat([angles, angles], dim=-1).to(dtype=q.dtype)
    cos = angles.cos()[None, None, :, :]
    sin = angles.sin()[None, None, :, :]
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


class RoPESelfAttention(nn.Module):
    def __init__(self, d_model=512, head_dim=32, dropout=0.0):
        super().__init__()
        if d_model % head_dim != 0:
            raise ValueError("d_model must be divisible by head_dim")
        self.head_dim = head_dim
        self.num_heads = d_model // head_dim
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.output_projection = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, attention_bias=None):
        batch_size, sequence_length, d_model = x.shape
        qkv = self.qkv(x).view(batch_size, sequence_length, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        q, k = apply_rope(q, k)
        if attention_bias is not None:
            attention_bias = attention_bias.to(device=q.device, dtype=q.dtype)
        attended = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attention_bias,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        attended = attended.transpose(1, 2).contiguous().view(batch_size, sequence_length, d_model)
        return self.output_projection(attended)


class RoPETransformerBlock(nn.Module):
    def __init__(self, d_model=512, head_dim=32, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.attention_norm = nn.LayerNorm(d_model)
        self.attention = RoPESelfAttention(d_model=d_model, head_dim=head_dim, dropout=dropout)
        self.mlp_norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio * d_model),
            nn.GELU(),
            nn.Linear(mlp_ratio * d_model, d_model),
        )

    def forward(self, x, attention_bias=None):
        x = x + self.attention(self.attention_norm(x), attention_bias=attention_bias)
        x = x + self.mlp(self.mlp_norm(x))
        return x


class OneHotRoPEContactPredictor(nn.Module):
    def __init__(
        self,
        d_model=256,
        head_dim=64,
        num_layers=8,
        temperature=0.1,
        penalize_noncanonical_pairs=False,
        noncanonical_logit_penalty=-3.0,
        dropout=0.0,
        use_sequence_conv_blocks=True,
        use_pair_metadata_features=False,
        relative_position_feature_dim=32,
        penalize_short_range_pairs=True,
        short_range_pair_max_distance=4,
        short_range_logit_penalty=-3.0,
        device=None,
    ):
        super().__init__()
        self.temperature = temperature
        self.d_model = d_model
        self.head_dim = head_dim
        if d_model % head_dim != 0:
            raise ValueError("d_model must be divisible by head_dim")
        self.num_heads = d_model // head_dim
        self.penalize_noncanonical_pairs = penalize_noncanonical_pairs
        self.noncanonical_logit_penalty = noncanonical_logit_penalty
        self.use_sequence_conv_blocks = use_sequence_conv_blocks
        self.use_pair_metadata_features = use_pair_metadata_features
        self.relative_position_feature_dim = relative_position_feature_dim
        self.penalize_short_range_pairs = penalize_short_range_pairs
        self.short_range_pair_max_distance = short_range_pair_max_distance
        self.short_range_logit_penalty = short_range_logit_penalty
        self.device_name = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.transformer_dtype = torch.float32
        self.input_projection = nn.Linear(4, d_model)
        self.transformer = nn.ModuleList(
            [
                RoPETransformerBlock(d_model=d_model, head_dim=head_dim, dropout=dropout)
                for _ in range(num_layers)
            ]
        )
        self.sequence_conv_blocks = nn.ModuleList(
            [ConvNeXtV2Block1D(d_model) for _ in range(num_layers)]
        ) if use_sequence_conv_blocks else nn.ModuleList()
        self.output_norm = nn.LayerNorm(d_model)
        pair_metadata_dim = 0
        if use_pair_metadata_features:
            pair_metadata_dim = relative_position_feature_dim + len(UNORDERED_BASE_PAIR_TYPES)
        pair_feature_dim = 2 * d_model + pair_metadata_dim
        self.pair_feature_dim = pair_feature_dim
        self.pair_conv_net = nn.Sequential(
            ConvNeXtV2Block(pair_feature_dim),
            ConvNeXtV2Block(pair_feature_dim),
            # ConvNeXtV2Block(pair_feature_dim),
            # ConvNeXtV2Block(pair_feature_dim),
        )
        self.pair_score_projection = nn.Conv2d(pair_feature_dim, 1, kernel_size=1)
        self.to(self.device_name)

    @property
    def device(self):
        return torch.device(self.device_name)

    def head_parameters(self):
        yield from self.parameters()

    def forward(self, sequence: str):
        canonical_pair_mask = None
        attention_bias = None
        if self.penalize_noncanonical_pairs:
            canonical_pair_mask = build_canonical_pair_mask(sequence, device=self.device)
            attention_pair_mask = canonical_pair_mask.clone()
            attention_pair_mask.fill_diagonal_(True)
            attention_bias = (~attention_pair_mask).to(dtype=self.transformer_dtype)[None, None, :, :] * self.noncanonical_logit_penalty

        base_features = sequence_to_one_hot(sequence, device=self.device, dtype=self.transformer_dtype)
        node_features = self.input_projection(base_features).unsqueeze(0)
        for block_index, transformer_block in enumerate(self.transformer):
            node_features = transformer_block(node_features, attention_bias=attention_bias)
            if self.use_sequence_conv_blocks:
                node_features = self.sequence_conv_blocks[block_index](node_features)
        node_features = self.output_norm(node_features).squeeze(0)
        sequence_length = node_features.shape[0]
        left_features = node_features[:, None, :].expand(sequence_length, sequence_length, -1)
        right_features = node_features[None, :, :].expand(sequence_length, sequence_length, -1)
        pair_feature_parts = [left_features, right_features]
        if self.use_pair_metadata_features:
            pair_feature_parts.append(
                relative_position_sinusoidal_features(
                    sequence_length,
                    self.relative_position_feature_dim,
                    device=self.device,
                    dtype=node_features.dtype,
                )
            )
            pair_feature_parts.append(
                unordered_base_pair_one_hot_features(
                    sequence,
                    device=self.device,
                    dtype=node_features.dtype,
                )
            )
        pair_features = torch.cat(pair_feature_parts, dim=-1).permute(2, 0, 1).unsqueeze(0).contiguous()
        pair_features = self.pair_conv_net(pair_features)
        pair_features = 0.5 * (pair_features + pair_features.transpose(-1, -2))
        logits = self.pair_score_projection(pair_features).squeeze(0).squeeze(0) / self.temperature
        if self.penalize_noncanonical_pairs:
            logits = add_noncanonical_logit_penalty(logits, canonical_pair_mask, self.noncanonical_logit_penalty)
        if self.penalize_short_range_pairs:
            logits = add_short_range_logit_penalty(
                logits,
                max_distance=self.short_range_pair_max_distance,
                penalty=self.short_range_logit_penalty,
            )
        return logits

    def valid_pair_mask(self, sequence: str, device=None):
        device = device or self.device
        valid_pair_mask = None
        if self.penalize_noncanonical_pairs:
            valid_pair_mask = build_canonical_pair_mask(sequence, device=device)
        if self.penalize_short_range_pairs:
            positions = torch.arange(len(sequence), device=device)
            long_range_mask = (positions[:, None] - positions[None, :]).abs() > self.short_range_pair_max_distance
            valid_pair_mask = long_range_mask if valid_pair_mask is None else valid_pair_mask & long_range_mask
        return valid_pair_mask

def contact_pair_loss_mask(n, device, valid_pair_mask=None):
    pair_mask = torch.triu(torch.ones((n, n), dtype=torch.bool, device=device), diagonal=1)
    if valid_pair_mask is not None:
        valid_pair_mask = valid_pair_mask.to(device=device, dtype=torch.bool)
        if valid_pair_mask.shape != (n, n):
            raise ValueError(f"Expected valid pair mask shape {(n, n)}, got {tuple(valid_pair_mask.shape)}")
        pair_mask = pair_mask & valid_pair_mask
    return pair_mask


def contact_binary_cross_entropy_loss(logits, target_adjacency, valid_pair_mask=None, positive_weight=10.0):
    logits = logits.float()
    target_adjacency = target_adjacency.to(device=logits.device, dtype=logits.dtype)
    n = target_adjacency.shape[0]
    if logits.shape != (n, n):
        raise ValueError(f"Expected logits shape {(n, n)}, got {tuple(logits.shape)}")
    target_adjacency = target_adjacency.clone()
    target_adjacency.fill_diagonal_(0.0)
    pair_mask = contact_pair_loss_mask(n, logits.device, valid_pair_mask=valid_pair_mask)
    pos_weight = torch.tensor(positive_weight, device=logits.device, dtype=logits.dtype)
    element_loss = F.binary_cross_entropy_with_logits(
        logits,
        target_adjacency,
        pos_weight=pos_weight,
        reduction="none",
    )
    masked_loss = element_loss[pair_mask]
    if masked_loss.numel() == 0:
        return element_loss.sum() * 0.0
    return masked_loss.mean()


def contact_dice_loss(logits, target_adjacency, valid_pair_mask=None, smooth=1.0, eps=1e-7):
    logits = logits.float()
    target_adjacency = target_adjacency.to(device=logits.device, dtype=logits.dtype)
    n = target_adjacency.shape[0]
    if logits.shape != (n, n):
        raise ValueError(f"Expected logits shape {(n, n)}, got {tuple(logits.shape)}")
    target_adjacency = target_adjacency.clone()
    target_adjacency.fill_diagonal_(0.0)
    pair_mask = contact_pair_loss_mask(n, logits.device, valid_pair_mask=valid_pair_mask)
    probabilities = torch.sigmoid(logits)[pair_mask]
    targets = target_adjacency[pair_mask]
    if probabilities.numel() == 0:
        return logits.sum() * 0.0
    intersection = (probabilities * targets).sum()
    denominator = probabilities.sum() + targets.sum()
    dice_score = (2.0 * intersection + smooth) / (denominator + smooth + eps)
    return 1.0 - dice_score


def apply_grokfast_ema(model, grokfast_state=None, alpha=0.98, lamb=2.0, apply=True):
    if grokfast_state is None:
        grokfast_state = {}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad or parameter.grad is None:
            continue
        grad = parameter.grad.detach()
        if not torch.isfinite(grad).all():
            grokfast_state.pop(name, None)
            parameter.grad = None
            continue
        if name not in grokfast_state or not torch.isfinite(grokfast_state[name]).all():
            grokfast_state[name] = grad.clone()
        else:
            grokfast_state[name].mul_(alpha).add_(grad, alpha=1.0 - alpha)
        if apply:
            parameter.grad.add_(grokfast_state[name], alpha=lamb)
    return grokfast_state


In [8]:

max_sequence_length = 2000
rna_dataset = RNADotBracketDataset(train_records, max_length=max_sequence_length, min_contacts=0)
train_loader = DataLoader(
    rna_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=collate_single_record,
    num_workers=16,
    generator=torch.Generator().manual_seed(20260702),
)
validation_dataset = RNADotBracketDataset(validation_records, max_length=max_sequence_length, min_contacts=1)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_single_record,
    num_workers=16,
)

lengths = np.array([len(record.sequence) for record in rna_dataset.records])
contacts = np.array([len(record.contacts) for record in rna_dataset.records])
print(f"Dataset records: {len(rna_dataset):,}")
print(f"Validation records: {len(validation_dataset):,}")
print(f"Length range: {lengths.min()}-{lengths.max()} nt | median={np.median(lengths):.0f}")
print(f"Contact range: {contacts.min()}-{contacts.max()} | median={np.median(contacts):.0f}")


penalize_noncanonical_pairs = True
noncanonical_logit_penalty = 0.0
print_tensor_core_status()
print_attention_backend_status()
print(f"Penalize noncanonical pairs: {penalize_noncanonical_pairs} ({noncanonical_logit_penalty})")
for stale_name in ("optimizer", "contact_model", "grokfast_state"):
    stale_object = globals().pop(stale_name, None)
    if stale_object is not None:
        del stale_object
release_cuda_memory()
contact_model = OneHotRoPEContactPredictor(
    temperature=0.3,
    penalize_noncanonical_pairs=penalize_noncanonical_pairs,
    noncanonical_logit_penalty=noncanonical_logit_penalty,
    use_sequence_conv_blocks=False,
    use_pair_metadata_features=True,
    relative_position_feature_dim=32,
    penalize_short_range_pairs=True,
    short_range_pair_max_distance=4,
    short_range_logit_penalty=-3.0,
)
optimizer = torch.optim.AdamW(
    (parameter for parameter in contact_model.parameters() if parameter.requires_grad),
    lr=3e-4,
    weight_decay=1e-4,
)
print(f"Model device: {contact_model.device}")
print(f"Model dim: {contact_model.d_model} | head_dim: {contact_model.head_dim} | heads: {contact_model.num_heads}")
print(f"Pair channels: {contact_model.pair_feature_dim}")
print(f"Transformer dtype: {contact_model.transformer_dtype} | head/loss=float32")
trainable_parameters = sum(parameter.numel() for parameter in contact_model.parameters() if parameter.requires_grad)
total_parameters = sum(parameter.numel() for parameter in contact_model.parameters())
print(f"Trainable parameters: {trainable_parameters:,} / {total_parameters:,}")

max_train_steps = 1000000
gradient_accumulation_steps = 4
show_every = 1000
validation_threshold = 0.5
wandb_log_every = 100
mask_invalid_pairs_in_loss_and_eval = True
dice_loss_weight = 1.0
use_grokfast = True
grokfast_alpha = 0.98
grokfast_lamb = 2.0
grokfast_warmup_steps = 100
grokfast_state = None
print(
    f"Grokfast EMA enabled: {use_grokfast} alpha={grokfast_alpha} "
    f"lambda={grokfast_lamb} warmup_steps={grokfast_warmup_steps}"
)
wandb_config = {
    "max_sequence_length": max_sequence_length,
    "train_records": len(rna_dataset),
    "validation_records": len(validation_dataset),
    "max_train_steps": max_train_steps,
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "validation_threshold": validation_threshold,
    "wandb_log_every": wandb_log_every,
    "mask_invalid_pairs_in_loss_and_eval": mask_invalid_pairs_in_loss_and_eval,
    "dice_loss_weight": dice_loss_weight,
    "model_dim": contact_model.d_model,
    "head_dim": contact_model.head_dim,
    "num_heads": contact_model.num_heads,
    "penalize_noncanonical_pairs": penalize_noncanonical_pairs,
    "noncanonical_logit_penalty": noncanonical_logit_penalty,
    "use_sequence_conv_blocks": contact_model.use_sequence_conv_blocks,
    "use_pair_metadata_features": contact_model.use_pair_metadata_features,
    "relative_position_feature_dim": contact_model.relative_position_feature_dim,
    "pair_feature_dim": contact_model.pair_feature_dim,
    "penalize_short_range_pairs": contact_model.penalize_short_range_pairs,
    "short_range_pair_max_distance": contact_model.short_range_pair_max_distance,
    "short_range_logit_penalty": contact_model.short_range_logit_penalty,
    "use_grokfast": use_grokfast,
    "grokfast_alpha": grokfast_alpha,
    "grokfast_lamb": grokfast_lamb,
    "grokfast_warmup_steps": grokfast_warmup_steps,
}
if wandb.run is None:
    wandb_run = wandb.init(project="rna-secondary", config=wandb_config)
else:
    wandb_run = wandb.run
    wandb_run.config.update(wandb_config, allow_val_change=True)
loss_history = []
bce_loss_history = []
dice_loss_history = []
f1_history = []
validation_f1_history = []
validation_loss_history = []
validation_bce_loss_history = []
validation_dice_loss_history = []
ema_loss = None
ema_f1 = None
loader_iter = iter(train_loader)
epoch_index = 1
epoch_batch_count = 0
epoch_progress = tqdm(total=len(train_loader), desc=f"epoch {epoch_index}")
optimizer.zero_grad(set_to_none=True)


def evaluate_validation_metrics(model, data_loader, threshold=0.5, mask_invalid_pairs=False, dice_weight=2.0):
    was_training = model.training
    model.eval()
    total_f1 = 0.0
    total_loss = 0.0
    total_bce_loss = 0.0
    total_dice_loss = 0.0
    count = 0
    with torch.no_grad():
        for batch in tqdm(data_loader, total=len(data_loader), desc="validation", leave=False):
            logits = model(batch["sequence"])
            target_adjacency = batch["target_adjacency"].to(model.device)
            valid_pair_mask = model.valid_pair_mask(batch["sequence"], device=model.device) if mask_invalid_pairs else None
            bce_loss = contact_binary_cross_entropy_loss(
                logits,
                target_adjacency,
                valid_pair_mask=valid_pair_mask,
            ).float()
            dice_loss = contact_dice_loss(
                logits,
                target_adjacency,
                valid_pair_mask=valid_pair_mask,
            ).float()
            loss = bce_loss + dice_weight * dice_loss
            f1_value = f1_rna_llm_folding(
                target_adjacency,
                logits,
                len(batch["sequence"]),
                th=threshold,
                valid_pair_mask=valid_pair_mask,
            )
            total_f1 += f1_value
            total_loss += float(loss.detach().cpu())
            total_bce_loss += float(bce_loss.detach().cpu())
            total_dice_loss += float(dice_loss.detach().cpu())
            count += 1
            del logits, target_adjacency, valid_pair_mask, bce_loss, dice_loss, loss
    if was_training:
        model.train()
    return {
        "f1": total_f1 / max(count, 1),
        "loss": total_loss / max(count, 1),
        "bce_loss": total_bce_loss / max(count, 1),
        "dice_loss": total_dice_loss / max(count, 1),
    }, count

for step in range(1, max_train_steps + 1):
    contact_model.train()
    accumulated_loss = 0.0
    accumulated_bce_loss = 0.0
    accumulated_dice_loss = 0.0
    accumulated_f1 = 0.0
    last_batch = None
    last_logits = None
    actual_accumulations = 0
    pending_epoch_validation = False

    for accumulation_index in range(gradient_accumulation_steps):
        try:
            batch = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader)
            batch = next(loader_iter)

        logits = contact_model(batch["sequence"])
        target_adjacency = batch["target_adjacency"].to(contact_model.device)
        valid_pair_mask = contact_model.valid_pair_mask(batch["sequence"], device=contact_model.device) if mask_invalid_pairs_in_loss_and_eval else None
        bce_loss = contact_binary_cross_entropy_loss(
            logits,
            target_adjacency,
            valid_pair_mask=valid_pair_mask,
        ).float()
        dice_loss = contact_dice_loss(
            logits,
            target_adjacency,
            valid_pair_mask=valid_pair_mask,
        ).float()
        loss = bce_loss + dice_loss_weight * dice_loss
        (loss / gradient_accumulation_steps).backward()
        with torch.no_grad():
            f1_value = f1_rna_llm_folding(
                target_adjacency,
                logits.detach(),
                len(batch["sequence"]),
                th=0.5,
                valid_pair_mask=valid_pair_mask,
            )

        accumulated_loss += float(loss.detach().cpu())
        accumulated_bce_loss += float(bce_loss.detach().cpu())
        accumulated_dice_loss += float(dice_loss.detach().cpu())
        accumulated_f1 += f1_value
        last_batch = batch
        last_logits = logits.detach().cpu()
        actual_accumulations += 1
        epoch_batch_count += 1
        epoch_progress.update(1)
        del logits, target_adjacency, valid_pair_mask, bce_loss, dice_loss, loss

        if epoch_batch_count >= len(train_loader):
            pending_epoch_validation = True
            epoch_batch_count = 0
            loader_iter = iter(train_loader)
            break

    if actual_accumulations == 0:
        continue
    if actual_accumulations < gradient_accumulation_steps:
        gradient_scale = gradient_accumulation_steps / actual_accumulations
        for parameter in contact_model.parameters():
            if parameter.grad is not None:
                parameter.grad.mul_(gradient_scale)
    raw_grad_norm = torch.nn.utils.clip_grad_norm_(
        contact_model.head_parameters(),
        max_norm=1.0,
        error_if_nonfinite=False,
    )
    final_grad_norm = raw_grad_norm
    optimizer_step_skipped = not torch.isfinite(raw_grad_norm).item()
    grokfast_active = use_grokfast and step > grokfast_warmup_steps and not optimizer_step_skipped
    grokfast_updating = use_grokfast and not optimizer_step_skipped
    if grokfast_updating:
        grokfast_state = apply_grokfast_ema(
            contact_model,
            grokfast_state=grokfast_state,
            alpha=grokfast_alpha,
            lamb=grokfast_lamb,
            apply=grokfast_active,
        )
        if grokfast_active:
            final_grad_norm = torch.nn.utils.clip_grad_norm_(
                contact_model.head_parameters(),
                max_norm=1.0,
                error_if_nonfinite=False,
            )
            optimizer_step_skipped = not torch.isfinite(final_grad_norm).item()
    if optimizer_step_skipped:
        grokfast_state = None
    else:
        optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    mean_loss = accumulated_loss / actual_accumulations
    mean_bce_loss = accumulated_bce_loss / actual_accumulations
    mean_dice_loss = accumulated_dice_loss / actual_accumulations
    mean_f1 = accumulated_f1 / actual_accumulations
    ema_loss = mean_loss if ema_loss is None else 0.99 * ema_loss + 0.01 * mean_loss
    ema_f1 = mean_f1 if ema_f1 is None else 0.99 * ema_f1 + 0.01 * mean_f1
    loss_history.append(mean_loss)
    bce_loss_history.append(mean_bce_loss)
    dice_loss_history.append(mean_dice_loss)
    f1_history.append(mean_f1)
    if step % wandb_log_every == 0:
        wandb.log(
            {
                "train/loss": mean_loss,
                "train/bce_loss": mean_bce_loss,
                "train/dice_loss": mean_dice_loss,
                "train/f1": mean_f1,
            },
            step=step,
        )
    epoch_progress.set_postfix(
        loss=f"{mean_loss:.4f}",
        bce=f"{mean_bce_loss:.4f}",
        dice=f"{mean_dice_loss:.4f}",
        ema_loss=f"{ema_loss:.4f}",
        ema_f1=f"{ema_f1:.4f}",
        grad=f"{float(raw_grad_norm.detach().cpu()):.3g}",
        final_grad=f"{float(final_grad_norm.detach().cpu()):.3g}",
        grokfast=int(grokfast_active),
        skipped=int(optimizer_step_skipped),
    )

    # if step == 1 or step % 10 == 0:
    #     record = last_batch["record"]
    #     print(
    #         f"step={step:04d} ema_loss={ema_loss:.4f} ema_f1={ema_f1:.4f} "
    #         f"accumulations={actual_accumulations}/{gradient_accumulation_steps} "
    #         f"last_record={record.name} length={len(record.sequence)} contacts={len(record.contacts)}"
    #     )

    if pending_epoch_validation:
        epoch_progress.close()
        validation_metrics, validation_count = evaluate_validation_metrics(
            contact_model,
            validation_loader,
            threshold=validation_threshold,
            mask_invalid_pairs=mask_invalid_pairs_in_loss_and_eval,
            dice_weight=dice_loss_weight,
        )
        validation_f1 = validation_metrics["f1"]
        validation_loss = validation_metrics["loss"]
        validation_bce_loss = validation_metrics["bce_loss"]
        validation_dice_loss = validation_metrics["dice_loss"]
        validation_f1_history.append((step, validation_f1))
        validation_loss_history.append((step, validation_loss))
        validation_bce_loss_history.append((step, validation_bce_loss))
        validation_dice_loss_history.append((step, validation_dice_loss))
        wandb.log(
            {
                "validation/loss": validation_loss,
                "validation/bce_loss": validation_bce_loss,
                "validation/dice_loss": validation_dice_loss,
                "validation/f1": validation_f1,
            },
            step=step,
        )
        print(
            f"validation loss={validation_loss:.4f} bce={validation_bce_loss:.4f} "
            f"dice={validation_dice_loss:.4f} f1={validation_f1:.4f} "
            f"threshold={validation_threshold:g} "
            f"records={validation_count:,}"
        )
        epoch_index += 1
        epoch_progress = tqdm(total=len(train_loader), desc=f"epoch {epoch_index}")
    

Dataset records: 10,814
Validation records: 1,299
Length range: 33-498 nt | median=105
Contact range: 0-166 | median=25
CUDA device: NVIDIA H100 NVL | capability=(9, 0)
BF16 supported: True
TF32 matmul enabled: True
TF32 cuDNN enabled: True
float32 matmul precision: high
flash-attn package installed: False
PyTorch SDPA available: True
PyTorch flash SDPA enabled: True
PyTorch memory-efficient SDPA enabled: True
PyTorch math SDPA enabled: True
Penalize noncanonical pairs: True (0.0)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/azureuser/.netrc.


Model device: cuda
Model dim: 256 | head_dim: 64 | heads: 4
Pair channels: 554
Transformer dtype: torch.float32 | head/loss=float32
Trainable parameters: 11,303,103 / 11,303,103
Grokfast EMA enabled: True alpha=0.98 lambda=2.0 warmup_steps=100


wandb: Currently logged in as: odusseys to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


epoch 1: 100%|██████████| 10814/10814 [07:08<00:00, 25.26it/s, bce=0.0704, dice=0.5086, ema_f1=0.4620, ema_loss=0.8143, final_grad=1.05, grad=2.99, grokfast=1, loss=0.5790, skipped=0]  


...

validation loss=0.6849 bce=0.2860 dice=0.3988 f1=0.6209 threshold=0.5 records=1,299


epoch 16:  84%|████████▎ | 9044/10814 [06:00<01:00, 29.24it/s, bce=0.2184, dice=0.3220, ema_f1=0.7895, ema_loss=0.3383, final_grad=1.04, grad=4.16, grokfast=1, loss=0.5404, skipped=0]  

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7afceead1a10>> (for post_run_cell), with arguments args (<ExecutionResult object at 7b017c547090, execution_count=8 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7b017c22b490, raw_cell="
max_sequence_length = 2000
rna_dataset = RNADotBr.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bazure/home/azureuser/physics/rna/rna_secondary.ipynb#Y151sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost